# ⏱️ Survival Analysis
**ISLP Chapter 11 · Pattern Recognition for the Rest of Us**

> Survival analysis answers questions like: how long until a patient relapses? When will a customer churn? It handles a unique challenge — censored observations: people who haven't experienced the event yet when the study ends.

### What you'll learn
- Censoring: why standard regression fails for time-to-event data
- Kaplan-Meier estimator: non-parametric survival curve
- Log-rank test: comparing survival between groups
- Cox proportional hazards model: regression for survival data
- Hazard ratio interpretation

### Dataset: ROSSI recidivism + simulated patient data
### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
!pip install lifelines -q
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.datasets import load_rossi
import warnings; warnings.filterwarnings('ignore')

# Rossi recidivism dataset: time until re-arrest after release from prison
rossi = load_rossi()
print(f'Rossi dataset: {rossi.shape}')
print(rossi.head())
print(f"\nEvent rate: {rossi['arrest'].mean():.1%} arrested")
print(f"Duration range: {rossi['week'].min()}–{rossi['week'].max()} weeks")

## 🎯 Part 1 — What is Censoring?

In time-to-event data:
- Some subjects experience the event (arrested, died, churned)
- Others are **censored** — we know they survived UNTIL a certain time, but not what happened after
  - Study ended before they experienced the event
  - Lost to follow-up
  - Withdrew from study

**Why standard regression fails:** A censored observation isn't a missing value — it contains real information (the person survived for at least X weeks). Ignoring it or treating it as missing biases estimates.

**Survival analysis** handles censoring properly.

In [ ]:
# Visualize censoring
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Show 20 individual timelines
np.random.seed(42)
sample_idx = np.random.choice(len(rossi), 25, replace=False)
for i, idx in enumerate(sample_idx):
    row = rossi.iloc[idx]
    color = '#e85d20' if row['arrest']==1 else '#1e5fa8'
    marker = 'x' if row['arrest']==1 else 'o'
    axes[0].plot([0, row['week']], [i, i], color=color, lw=1.5, alpha=0.8)
    axes[0].plot(row['week'], i, marker=marker, color=color, markersize=7)
axes[0].set_xlabel('Weeks'); axes[0].set_ylabel('Subject')
axes[0].set_title('Event timelines\n× = arrested (event), o = censored')
from matplotlib.lines import Line2D
legend = [Line2D([0],[0],color='#e85d20',marker='x',label='Arrested'),
          Line2D([0],[0],color='#1e5fa8',marker='o',label='Censored')]
axes[0].legend(handles=legend)

# Histogram of observation times by event status
for arrested, color, label in [(1,'#e85d20','Arrested'),(0,'#1e5fa8','Censored')]:
    subset = rossi[rossi['arrest']==arrested]['week']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=f'{label} (n={len(subset)})', edgecolor='white')
axes[1].set_xlabel('Weeks'); axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Follow-up Times')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Kaplan-Meier survival curve — overall
kmf = KaplanMeierFitter()
kmf.fit(rossi['week'], event_observed=rossi['arrest'], label='All subjects')

fig, ax = plt.subplots(figsize=(9, 5))
kmf.plot_survival_function(ax=ax, color='#1e5fa8', linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release')
ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier Survival Curve — Recidivism Study')
ax.axhline(0.5, color='#888', lw=1, ls='--', label='Median survival')
median = kmf.median_survival_time_
if not np.isinf(median):
    ax.axvline(median, color='#888', lw=1, ls='--')
    ax.text(median+1, 0.52, f'Median = {median:.0f} weeks', fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()
print(f'\nMedian survival time: {kmf.median_survival_time_:.1f} weeks')
print(f'52-week survival estimate: {kmf.survival_function_at_times([52]).values[0]:.3f}')

In [ ]:
# Compare groups: financial aid vs no financial aid
fig, ax = plt.subplots(figsize=(9, 5))
for group, color, label in [(1,'#1a7a45','Received financial aid'),(0,'#e85d20','No financial aid')]:
    kmf_g = KaplanMeierFitter()
    mask = rossi['fin']==group
    kmf_g.fit(rossi.loc[mask,'week'], event_observed=rossi.loc[mask,'arrest'], label=label)
    kmf_g.plot_survival_function(ax=ax, color=color, linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release'); ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier by Financial Aid Status')
ax.legend()
plt.tight_layout(); plt.show()

# Log-rank test
results = logrank_test(
    rossi.loc[rossi['fin']==1,'week'], rossi.loc[rossi['fin']==0,'week'],
    event_observed_A=rossi.loc[rossi['fin']==1,'arrest'],
    event_observed_B=rossi.loc[rossi['fin']==0,'arrest']
)
print(f'Log-rank test: p-value = {results.p_value:.4f}')
print('Financial aid significantly reduces recidivism' if results.p_value < 0.05 else 'No significant difference')

In [ ]:
# Cox Proportional Hazards Model
cox_features = ['fin','age','race','wexp','mar','paro','prio']
cph = CoxPHFitter()
cph.fit(rossi[cox_features + ['week','arrest']], duration_col='week', event_col='arrest')
cph.print_summary()

# Plot hazard ratios
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.set_title('Cox Model: Hazard Ratios (HR > 1 = higher risk of arrest)')
ax.axvline(0, color='black', lw=0.8, ls='--')
plt.tight_layout(); plt.show()
print('\n📌 HR < 0 (log scale) = protective factor, HR > 0 = risk factor')
print(f"   'prio' (prior arrests): HR = {np.exp(cph.params_['prio']):.2f} — each prior arrest increases risk by {(np.exp(cph.params_['prio'])-1)*100:.0f}%")

In [ ]:
# Exercise workspace
# Task 1: Compare survival curves by age group (< 30 vs >= 30)
# YOUR CODE HERE

# Task 2: Build a Cox model using only the 3 most significant predictors
# Compare its concordance index to the full model
# YOUR CODE HERE

# Task 3: Predict individual survival curves for two specific profiles:
# Profile A: young (25), prior arrests=0, financial aid
# Profile B: older (40), prior arrests=3, no financial aid
profile_A = pd.DataFrame({'fin':1,'age':25,'race':1,'wexp':1,'mar':0,'paro':1,'prio':0}, index=[0])
profile_B = pd.DataFrame({'fin':0,'age':40,'race':1,'wexp':0,'mar':0,'paro':0,'prio':3}, index=[0])
# YOUR CODE HERE

In [ ]:
# @title 📝 Quiz — Survival Analysis
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is censoring in survival analysis?
# @markdown **Q2:** What does the Kaplan-Meier curve show?
# @markdown **Q3:** What does the log-rank test compare?
# @markdown **Q4:** What does a hazard ratio of 2.0 mean in a Cox model?
# @markdown **Q5:** Why can't you use standard linear regression for time-to-event outcomes?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_="Survival Analysis"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


## 📚 Further Reading
- [ISLP Ch 11](https://intro-stat-learning.github.io/ISLP/) — Survival analysis
- [lifelines docs](https://lifelines.readthedocs.io/)
- [Cox model interpretation guide](https://lifelines.readthedocs.io/en/latest/Survival%20Regression.html)

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*